# 📓 Notebook 05 — Full Trading Agent

## รวมทุกอย่างจาก 4 Notebooks:
- MT5 data + SMC indicators (NB01)
- LLM + Vision (NB03)
- Agent + Tools (NB04)
- **Top-Down Analysis** (1W→1D→4H→15m→1m)
- **Interactive Chat + Auto Loop**

> ⚠️ ต้องมี `OPENAI_API_KEY` ใน `.env`

In [ ]:
import os, io, base64, time
import requests
from datetime import datetime
import pandas as pd, numpy as np
import smartmoneyconcepts as smc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from IPython.display import Image, display

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
TELEGRAM_BOT_TOKEN = os.getenv('TELEGRAM_BOT_TOKEN')
TELEGRAM_CHAT_ID = os.getenv('TELEGRAM_CHAT_ID')
print(f'✅ Ready | Model: {OPENAI_MODEL}')

In [ ]:
# ========== Core Functions ==========
plt.style.use('dark_background')

def _get_mt5(symbol='XAUUSD', tf='15m', count=500):
    try:
        import MetaTrader5 as mt5
        if not mt5.initialize(): return None
        tf_map = {'1m':mt5.TIMEFRAME_M1,'5m':mt5.TIMEFRAME_M5,'15m':mt5.TIMEFRAME_M15,
                  '1h':mt5.TIMEFRAME_H1,'4h':mt5.TIMEFRAME_H4,'1d':mt5.TIMEFRAME_D1,'1w':mt5.TIMEFRAME_W1}
        rates = mt5.copy_rates_from_pos(symbol, tf_map.get(tf,mt5.TIMEFRAME_M15), 0, count)
        mt5.shutdown()
        if rates is None: return None
        df = pd.DataFrame(rates); df['date']=pd.to_datetime(df['time'],unit='s')
        df=df.rename(columns={'tick_volume':'volume'})
        return df[['date','open','high','low','close','volume']].copy()
    except: return None

def _get_data(tf='15m', count=200):
    df = _get_mt5(tf=tf, count=count)
    if df is not None: return df
    if os.path.exists('sample_xauusd.csv'):
        return pd.read_csv('sample_xauusd.csv',parse_dates=['date']).tail(count).reset_index(drop=True)
    np.random.seed(42); base=2650; prices=base+np.cumsum(np.random.randn(count)*2.5)
    data=[{'date':pd.Timestamp('2025-01-01')+pd.Timedelta(minutes=15*i),'open':round(p+np.random.randn()*1.5,2),
           'high':round(max(p+np.random.randn()*1.5,p)+abs(np.random.randn())*3,2),
           'low':round(min(p+np.random.randn()*1.5,p)-abs(np.random.randn())*3,2),
           'close':round(p,2),'volume':int(abs(np.random.randn())*1000+500)} for i,p in enumerate(prices)]
    return pd.DataFrame(data)

def _compute(df):
    sl=20 if len(df)>=200 else 15 if len(df)>=100 else 10 if len(df)>=50 else max(5,len(df)//6)
    sw=smc.smc.swing_highs_lows(df,swing_length=sl)
    return {'swing':sw,'fvg':smc.smc.fvg(df),'bos_choch':smc.smc.bos_choch(df,sw,close_break=True),
            'ob':smc.smc.ob(df,sw),'liquidity':smc.smc.liquidity(df,sw,range_percent=0.02)}

def _summarize(df,ind,tf='15m'):
    last=df.iloc[-1]; f=ind['fvg']; b=ind['bos_choch']; o=ind['ob']; l=ind['liquidity']
    lines=[f'=== SMC {tf} (Price: {last["close"]:.2f}) ===']
    lines.append(f'FVG: Bull={((f["FVG"]==1)&f["MitigatedIndex"].isna()).sum()}, Bear={((f["FVG"]==-1)&f["MitigatedIndex"].isna()).sum()}')
    bm=b['BOS'].notna()
    if bm.any(): lb=b[bm].iloc[-1]; lines.append(f'BOS: {"Bull" if lb["BOS"]==1 else "Bear"} @ {lb["Level"]:.2f}')
    cm=b['CHOCH'].notna()
    if cm.any(): lc=b[cm].iloc[-1]; lines.append(f'CHOCH: {"Bull" if lc["CHOCH"]==1 else "Bear"} @ {lc["Level"]:.2f}')
    lines.append(f'OB: Bull={((o["OB"]==1)&o["MitigatedIndex"].isna()).sum()}, Bear={((o["OB"]==-1)&o["MitigatedIndex"].isna()).sum()}')
    lines.append(f'Liq: BSL={((l["Liquidity"]==1)&l["Swept"].isna()).sum()}, SSL={((l["Liquidity"]==-1)&l["Swept"].isna()).sum()}')
    return '\n'.join(lines)

def _chart_b64(df, ind, last_n=50):
    """สร้างกราฟ → base64"""
    s=max(0,len(df)-last_n); w=df.iloc[s:]; n=len(w)
    fw=ind['fvg'].iloc[s:]; ow=ind['ob'].iloc[s:]
    fig,ax=plt.subplots(figsize=(12,5),facecolor='#0C0E12'); ax.set_facecolor('#0C0E12')
    o,h,l,c=w['open'].values,w['high'].values,w['low'].values,w['close'].values
    for i in range(n):
        col='#77dd76' if c[i]>=o[i] else '#ff6962'
        ax.plot([i,i],[l[i],h[i]],color=col,lw=1)
        ax.add_patch(Rectangle((i-.3,min(o[i],c[i])),.6,max(abs(c[i]-o[i]),.1),facecolor=col,alpha=.8))
    for i in range(len(fw)):
        if pd.notna(fw['FVG'].iloc[i]) and fw['FVG'].iloc[i]!=0 and pd.isna(fw['MitigatedIndex'].iloc[i]):
            ax.add_patch(Rectangle((i,fw['Bottom'].iloc[i]),n-i,fw['Top'].iloc[i]-fw['Bottom'].iloc[i],facecolor='yellow',alpha=.2))
    for i in range(len(ow)):
        if pd.notna(ow['OB'].iloc[i]) and ow['OB'].iloc[i]!=0 and pd.isna(ow['MitigatedIndex'].iloc[i]):
            cc='purple' if ow['OB'].iloc[i]==1 else 'magenta'
            ax.add_patch(Rectangle((i,ow['Bottom'].iloc[i]),n-i,ow['Top'].iloc[i]-ow['Bottom'].iloc[i],facecolor=cc,alpha=.25))
    ax.set_xlim(-.5,n-.5); ax.set_ylim(l.min()*.999,h.max()*1.001)
    ax.tick_params(colors='white',labelsize=7); ax.grid(False); plt.tight_layout()
    buf=io.BytesIO(); plt.savefig(buf,format='png',dpi=100,bbox_inches='tight',facecolor='#0C0E12')
    buf.seek(0); plt.close()
    return f'data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}'

print('✅ Core functions ready')

# ========== Telegram Core ==========
def send_telegram_message(message: str):
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID: return False
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    try:
        requests.post(url, json={"chat_id": TELEGRAM_CHAT_ID, "text": message, "parse_mode": "HTML"})
        return True
    except: return False

def send_telegram_base64_photo(base64_str: str, caption: str=""):
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID: return False
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendPhoto"
    try:
        if "," in base64_str: base64_str = base64_str.split(",")[1]
        image_data = base64.b64decode(base64_str)
        files = {"photo": ("chart.png", image_data, "image/png")}
        requests.post(url, data={"chat_id": TELEGRAM_CHAT_ID, "caption": caption, "parse_mode": "HTML"}, files=files)
        return True
    except: return False


In [ ]:
# ========== SMC Tools ==========
@tool
def get_smc_analysis(timeframe: str = '15m', candle_count: int = 200) -> str:
    """ดึง XAUUSD + SMC indicators
    Args: timeframe: '1m'-'1w' | candle_count: จำนวนแท่ง"""
    try:
        df=_get_data(tf=timeframe,count=candle_count)
        return _summarize(df,_compute(df),timeframe)
    except Exception as e: return f'Error: {e}'

@tool
def get_multi_tf() -> str:
    """วิเคราะห์ 3 TF (15m,1h,4h)"""
    r=[]
    for tf in ['15m','1h','4h']:
        try:
            df=_get_data(tf=tf,count=200)
            r.append(f'\n--- {tf} ---\n{_summarize(df,_compute(df),tf)}')
        except Exception as e: r.append(f'\n{tf} Error: {e}')
    return '\n'.join(r)

@tool
def get_stats() -> str:
    """สถิติ XAUUSD"""
    df=_get_data(count=500)
    return f'Price:{df["close"].iloc[-1]:.2f} H:{df["high"].max():.2f} L:{df["low"].min():.2f}'

SMC_TOOLS = [get_smc_analysis, get_multi_tf, get_stats]
print(f'✅ {len(SMC_TOOLS)} tools')

In [ ]:
# ========== Full Agent ==========
SYSTEM = '''คุณคือ AI Trading Analyst เชี่ยวชาญ SMC/ICT วิเคราะห์ XAUUSD
หลักการ: FVG=POI, BOS=trend, CHOCH=reversal, OB=entry, Liq=target
ต้องมี confluence 2-3 signals | R:R>=1:2 | ไม่ชัด→HOLD
Mode: ADVISORY (แนะนำเท่านั้น) | ตอบไทย'''

llm = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, temperature=0.1)
prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM),
    MessagesPlaceholder(variable_name='chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder(variable_name='agent_scratchpad'),
])
agent = create_tool_calling_agent(llm, SMC_TOOLS, prompt)
executor = AgentExecutor(agent=agent, tools=SMC_TOOLS, verbose=False,
                         max_iterations=8, handle_parsing_errors=True)
chat_history = []
print('✅ Full Agent ready')

## 📊 Top-Down Analysis (1W→1D→4H→15m→1m)

In [ ]:
def topdown_analysis():
    frames = [('1w','WEEKLY Bias'),('1d','DAILY Structure'),('4h','4H OB/FVG'),
              ('15m','15M Entry'),('1m','1M Precision')]
    all_text=''; images=[]
    for tf,label in frames:
        print(f'  📡 {tf}...')
        try:
            df=_get_data(tf=tf,count=200); ind=_compute(df)
            all_text += f'\n[{label}]\n{_summarize(df,ind,tf)}\n'
            images.append({'label':f'{tf} ({label})','b64':_chart_b64(df,ind,50)})
        except Exception as e:
            all_text += f'\n[{label}] Error: {e}\n'
    
    content=[{'type':'text','text':f'ICT Top-Down Analysis:\n{all_text}\n\n'
              'Verdict: 1)Weekly Bias 2)Daily Confirm 3)4H Structure '
              '4)15m Signal 5)BUY/SELL/HOLD 6)Entry,SL,TP | ตอบไทย'}]
    for img in images:
        content.append({'type':'text','text':f'\nChart {img["label"]}:'})
        content.append({'type':'image_url','image_url':{'url':img['b64'],'detail':'high'}})
    
    v_llm = ChatOpenAI(model='gpt-4o',api_key=OPENAI_API_KEY,temperature=0.1,max_tokens=4096)
    return v_llm.invoke([HumanMessage(content=content)]).content

print('🔍 Top-Down Analysis (อาจใช้เวลา 30-60 วิ)...')
result = topdown_analysis()
print('\n' + '='*60)
print('🎯 TOP-DOWN VERDICT:')
print('='*60)
print(result)
print('📤 ส่งผลไปยัง Telegram...')
send_telegram_message(f'📊 *TOP-DOWN ANALYSIS VERDICT*\n\n{result}')
for img in images:
    send_telegram_base64_photo(img['b64'], caption=f'Chart: {img["label"]}')

## 💬 Interactive Chat

In [ ]:
def interactive_chat():
    history = []
    print('='*50)
    print('  🤖 SMC Agent — Interactive (พิมพ์ quit เพื่อออก)')
    print('='*50)
    while True:
        user = input('\n🧑 คุณ: ').strip()
        if not user: continue
        if user.lower() in ('quit','exit','q','ออก'):
            print('👋 Bye!'); break
        if user=='analyze': user='วิเคราะห์ XAUUSD 15m BUY/SELL/HOLD พร้อม Entry/SL/TP'
        elif user=='mtf': user='Multi-TF Analysis (15m,1h,4h) สรุป confluence'
        try:
            r = executor.invoke({'input':user,'chat_history':history})
            print(f'\n🤖: {r["output"]}')
            history.append({'role':'user','content':user})
            history.append({'role':'assistant','content':r['output']})
            if len(history)>10: history=history[-10:]
        except KeyboardInterrupt: print('\n⏹️ Stopped'); break
        except Exception as e: print(f'❌ {e}')

interactive_chat()

## 🔄 Auto Analysis Loop

In [ ]:
def auto_loop(interval=60, rounds=3):
    print(f'🔄 Auto Loop: ทุก {interval}s, {rounds} รอบ')
    hist=[]
    for i in range(1,rounds+1):
        print(f'\n--- รอบ #{i} ({datetime.now().strftime("%H:%M:%S")}) ---')
        try:
            r=executor.invoke({'input':f'[Auto#{i}] วิเคราะห์ XAUUSD 15m BUY/SELL/HOLD','chat_history':hist})
            print(r['output'])
            send_telegram_message(f"🔄 *Auto Loop #{i}*\n\n{r['output']}")
            hist.append({'role':'user','content':f'Auto#{i}'})
            hist.append({'role':'assistant','content':r['output']})
            if len(hist)>6: hist=hist[-6:]
        except KeyboardInterrupt: print('⏹️'); break
        except Exception as e: print(f'❌ {e}')
        if i<rounds: print(f'⏳ รอ {interval}s...'); time.sleep(interval)
    print('✅ จบ Auto Loop')

auto_loop(interval=60, rounds=3)

## 🎓 สรุปทั้ง 5 Notebooks

| # | ทักษะ |
|---|-------|
| 01 | MT5 data + SMC indicators + Chart |
| 02 | FVG, BOS, OB, Liquidity |
| 03 | LLM + Vision |
| 04 | Agent + Tools |
| 05 | Top-Down + Chat + Auto Loop |

**ทั้งหมดนี้ไม่ต้องรัน Backend เลย!** 🎉

---
> ⚠️ AI ไม่ใช่คำแนะนำทางการเงิน — ใช้เพื่อการศึกษาเท่านั้น